In [16]:
import pandas as pd
import numpy as np
from gensim.models.phrases import Phrases, Phraser
from gensim.models import Word2Vec
from gensim.test.utils import get_tmpfile
from gensim.models import KeyedVectors
from time import time 
import multiprocessing
from tqdm import tqdm
tqdm.pandas()

In [21]:
df = pd.read_csv("../data/05-cleaned.csv")
df['Text'] = df.progress_apply(lambda row: (row['Text'].split()), axis=1)
df = df['Text']


100%|██████████| 167033/167033 [00:03<00:00, 55125.24it/s]


In [22]:
sent = [row for row in df]
phrases = Phrases(sent, min_count=1, progress_per=50000)
bigram = Phraser(phrases)
sentences = bigram[sent]
sentences[1]

['utfallet',
 'pao',
 'denna',
 'poll',
 'blev',
 'som',
 'foeljer',
 'om',
 '24',
 'h',
 'vet',
 'vi',
 'om',
 'oever',
 'haelften_av',
 'de',
 'roestande',
 'kan',
 'sitt',
 'nya',
 'sverige',
 'eller_ej',
 '#svpolsteve_o',
 'wizard_of',
 'tweet_@gladsvartkvinna',
 'dec_31',
 '2019',
 'snabb',
 'poll',
 'vad',
 'kan',
 'ni',
 'om',
 'det',
 'nya',
 'sverige',
 '?',
 'hur_laong',
 'tid',
 'pao',
 'nya',
 'aoret_innan',
 'media',
 'rapporterat',
 'samtliga',
 'av',
 'nedan',
 '1',
 'oeverfallsvaoldtaekt',
 '2',
 'gruppvaoldtaekt',
 '3',
 'gaeng',
 'raonar_ensamt',
 'bo',
 'med',
 'show_this',
 'poll']

In [26]:
w2v_model = Word2Vec(min_count=3,
                     window=4,
                     size=300,
                     sample=1e-5, 
                     alpha=0.03, 
                     min_alpha=0.0007, 
                     negative=20,
                     workers=multiprocessing.cpu_count()-1)

In [27]:
start = time()
w2v_model.build_vocab(tqdm(sentences), progress_per=50000)


100%|██████████| 167033/167033 [00:15<00:00, 10959.24it/s]


In [28]:
start = time()

w2v_model.train(tqdm(sentences), total_examples=w2v_model.corpus_count, epochs=30, report_delay=1)

print('Time to train the model: {} mins'.format(round((time() - start) / 60, 2)))

w2v_model.init_sims(replace=True)

100%|██████████| 167033/167033 [00:17<00:00, 9766.80it/s]
Time to train the model: 8.42 mins


In [31]:
w2v_model.save("../models/2020word2vec.model")